In [2]:
import pandas as pd 
import numpy as np 

In [3]:
df=pd.read_csv("powerplant_data.csv")

In [4]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [5]:
#AT-temperature 
#v- vaccuum 
#AP-pressure 
#RH- humidity 
#PE-produced energy 

In [6]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [7]:
X=df.drop("PE",axis=1)
y=df["PE"]

In [8]:
X.head()

,AT,V,AP,RH
0,8.34,40.77,1010.84,90.01
1,23.64,58.49,1011.40,74.20
2,29.74,56.90,1007.15,41.91
3,19.07,49.69,1007.22,76.79
4,11.80,40.66,1017.13,97.20


In [9]:
#split data 
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [10]:
from sklearn.preprocessing import StandardScaler
scaler =StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [11]:
import torch 
import torch.nn as nn

In [12]:
X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [13]:
type(X_train_scaled)

numpy.ndarray

In [14]:
type(y_train)

pandas.core.series.Series

In [15]:
from torch.utils.data import TensorDataset,DataLoader
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
test_dataset=TensorDataset(X_test_tensor,y_test_tensor)

In [16]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

<h1>Deep Learning </h1>

In [17]:
#define an ANN model 
#in form of a class
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model=nn.Sequential(
            #1st hidden layer 
            nn.Linear(X_train.shape[1],6),
            nn.ReLU(),
            
            #2nd hidden
            nn.Linear(6,6),
            nn.ReLU(),
            
            #output layer 
            nn.Linear(6,1),
        )
    def forward(self,x):
        return self.model(x)

In [18]:
import torch.optim as optim
model=ANN()
#loss,optimizer
crietron=nn.MSELoss()
optimizer=optim.Adam(model.parameters())

In [19]:
#train the model 
train_losses=[]
valid_losses=[]
best_val_loss = float("inf")
epochs=100
for epoch in range(epochs):
    model.train()
    running_loss=0.0 #1 epoch training loss
    for xb,yb in train_loader:
        #xb- features of 1 batch 
        #yb-labels of 1 batch 
        optimizer.zero_grad()
        
        outputs=model(xb)#predicted outputs
        loss = crietron(outputs,yb) #compute loss
        loss.backward()#compute gradient
        optimizer.step() #params update 
        
        running_loss += loss.item() #loss is a tensor -> py float
        
    epoch_train_loss=running_loss/len(train_loader)
    train_losses.append(epoch_train_loss)

    #validation
    running_val_loss=0.0
    with torch.no_grad():
        for xb,yb in test_loader:
            outputs=model(xb)
            loss = crietron(outputs,yb)
            running_val_loss += loss
    epoch_val_loss=running_val_loss/len(test_loader)
    valid_losses.append(epoch_val_loss)
    print(f"epoch ${epoch+1}/{epochs} ==> train loss= ${epoch_train_loss} &val_loss= ${epoch_val_loss}")
    if epoch_val_loss<best_val_loss:
        best_val_loss=epoch_val_loss
        torch.save(model.state_dict(),"best_model.pt")

epoch $1/100 ==> train loss= $206530.50911458334 &val_loss= $205350.765625
epoch $2/100 ==> train loss= $201544.68697916667 &val_loss= $195035.734375
epoch $3/100 ==> train loss= $182788.63411458334 &val_loss= $166867.5625
epoch $4/100 ==> train loss= $144297.38076171876 &val_loss= $120313.4453125
epoch $5/100 ==> train loss= $95368.32700195312 &val_loss= $73171.1953125
epoch $6/100 ==> train loss= $55560.09768880208 &val_loss= $42329.7578125
epoch $7/100 ==> train loss= $33764.50196940104 &val_loss= $28159.8359375
epoch $8/100 ==> train loss= $24419.408744303386 &val_loss= $21803.810546875
epoch $9/100 ==> train loss= $19699.575594075523 &val_loss= $18056.224609375
epoch $10/100 ==> train loss= $16501.416467285155 &val_loss= $15012.10546875
epoch $11/100 ==> train loss= $13594.826696777343 &val_loss= $12185.9853515625
epoch $12/100 ==> train loss= $10857.231730143229 &val_loss= $9463.9521484375
epoch $13/100 ==> train loss= $8267.186938476563 &val_loss= $7010.80908203125
epoch $14/100

In [20]:
#evaluation
model.eval()
with torch.no_grad():
    train_preds=model(X_train_tensor)
    test_preds=model(X_test_tensor)
    train_mse_loss=crietron(train_preds,y_train_tensor)
    test_mse_loss=crietron(test_preds,y_test_tensor)
print("training MSE: ",train_mse_loss.item())
print("testing MSE: ",test_mse_loss.item())


training MSE:  21.14922523498535
testing MSE:  19.55259895324707


In [21]:
from sklearn.metrics import r2_score
print("r^2 score=",r2_score(y_test,test_preds))

r^2 score= 0.9316687102043394
